In [1]:
import sys
import os
import pandas as pd

src_path = os.path.abspath(os.path.join(os.path.dirname("__file__"), '..', 'src'))
if src_path not in sys.path:
    sys.path.append(src_path)

from dense_index_bge import DenseIndex as BGEDenseIndex
from dense_index_qwen import DenseIndex as QwenDenseIndex

from sentence_transformers import SentenceTransformer
import citation_utils
import metric_utils
import reranker_utils
import rrf
import hits_utils

/root/miniconda3/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/root/miniconda3/compiler_compat/ld: warning: librt.so.1, needed by /usr/local/cuda/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/root/miniconda3/compiler_compat/ld: warning: libpthread.so.0, needed by /usr/local/cuda/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/root/miniconda3/compiler_compat/ld: warning: libstdc++.so.6, needed by /usr/local/cuda/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/root/miniconda3/compiler_compat/ld: warning: libm.so.6, needed by /usr/local/cuda/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/root/miniconda3/compiler_compat/ld: /usr/local/cuda/lib64/libcufile.so: undefined reference to `std::runtime_error::~runtime_error()@GLIBCXX_3.4'
/root/miniconda3/compiler_compat/ld: /usr/local/cuda/lib64/libcufile.so: undefined reference to `__gxx_personality_v0@CXXABI_1.3

In [2]:
court_consideration_df = pd.read_csv("../data/court_considerations.csv")
court_consideration_d = dict(zip(court_consideration_df['citation'].tolist(), court_consideration_df['text'].tolist()))

law_df = pd.read_csv("../data/laws_de.csv")
law_d = dict(zip(law_df['citation'].tolist(), law_df['text'].tolist()))

court_doc = [{'citation':citation, 'text':text} for citation,text in zip(court_consideration_df['citation'].tolist(), court_consideration_df['text'].tolist())]
law_doc = [{'citation':citation, 'text':text} for citation,text in zip(law_df['citation'].tolist(), law_df['text'].tolist())]

valid_df = pd.read_csv("../data/valid_rewrite_001.csv")

print("data loaded.")

data loaded.


In [3]:
# Qwen-embedding

# from FlagEmbedding import FlagReranker

model1 = SentenceTransformer("/root/.cache/modelscope/hub/models/Qwen/Qwen3-Embedding-0___6B", model_kwargs={"torch_dtype": "float16"})

qwen_dense_index = QwenDenseIndex(model1, "../data/processed/_dense_court", court_doc)


DenseIndex.embeddings:  (2107648, 1024)


In [4]:
from FlagEmbedding import FlagReranker, BGEM3FlagModel
from sparse_index import SparseIndex as BGESparseIndex

model2 = BGEM3FlagModel('/root/.cache/modelscope/hub/models/BAAI/bge-m3', use_fp16=True, show_progress_bar=False)

bge_dense_index = BGEDenseIndex(model2, "/root/autodl-fs/bge-processed/_dense_sparse_court/", court_doc)
bge_sparse_index = BGESparseIndex(model2, "/root/autodl-fs/bge-processed/_dense_sparse_court/", court_doc)
bge_sparse_index.load()

DenseIndex.embeddings:  (2107648, 1024)


In [5]:
hits_l = []
gold_citations_l = []

valid_df = pd.read_csv("../data/valid_rewrite_001.csv")

for query, gold_citations in zip(valid_df['query2'].tolist(), valid_df['gold_citations'].tolist()):
    hits = qwen_dense_index.search_with_score(query, top_k=2000)
    gold_citations_l.append(gold_citations.split(";"))

    second_layer = citation_utils.compute_citation_score_with_sentence_pos(hits, decay="reciprocal")

    hits = []
    second_layer_request = []
    for citation, score in second_layer:
        if citation in court_consideration_d:
            hits.append({'citation':citation, 'text':court_consideration_d[citation]})
        if citation in law_d:
            hits.append({'citation':citation, 'text':law_d[citation]})

    hits_l.append(hits)
    
r = metric_utils.cal_recall(hits_l, gold_citations_l)

print("qwen recall:", r)

qwen recall: 0.42795203960678724


In [6]:
hits_l = []
gold_citations_l = []

valid_df = pd.read_csv("../data/valid_rewrite_001.csv")

for query, gold_citations in zip(valid_df['query2'].tolist(), valid_df['gold_citations'].tolist()):
    hits1 = bge_dense_index.search_with_score(query, top_k=1000)
    hits2 = bge_sparse_index.search_with_score(query, top_k=1000)
    hits_merge = hits_utils.merge_hits_with_score_l_by_max(hits1, hits2)
    print("hits_merge.len:", len(hits_merge))
    gold_citations_l.append(gold_citations.split(";"))

    hits = [hit for hit, score in hits_merge]
    
    second_layer = citation_utils.compute_citation_score_with_sentence_pos(hits_merge, decay="reciprocal")

    second_layer_request = []
    for citation, score in second_layer:
        if citation in court_consideration_d:
            hits.append({'citation':citation, 'text':court_consideration_d[citation]})
        if citation in law_d:
            hits.append({'citation':citation, 'text':law_d[citation]})

    hits_l.append(hits)
    
r = metric_utils.cal_recall(hits_l, gold_citations_l)

print("bge recall:", r)

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


hits_merge.len: 1603
hits_merge.len: 1860
hits_merge.len: 1897
hits_merge.len: 1739
hits_merge.len: 1773
hits_merge.len: 1759
hits_merge.len: 1899
hits_merge.len: 1887
hits_merge.len: 1817
hits_merge.len: 1880
bge recall: 0.5365915445587224
